# Week 9 Participation Activity: Image Site Downloader

**Qingan Xu**

**Task** (Chapter 13 practice program): write a program that goes to a photo-sharing site like Flickr or Imgur, searches for a category of photos, and downloads all of the resulting images.

### How the program works

1. **Search.** Flickr's search page relies heavily on JavaScript and loads more results as you scroll, so scraping it with `requests` and Beautiful Soup breaks easily. Chapter 13 recommends using a site's API when it has one, because an API changes far less often than the site's pages. Flickr has a free public photo feed that needs no account or API key: give it a tag and it returns up to 20 of the most recent public photos with that tag. The program requests the feed as JSON and reads it with `json.loads()`, the same way the chapter reads the OpenWeather API.
2. **Extract text.** Each photo in the feed comes with a short block of HTML. Beautiful Soup's `select()` and `get_text()` pull the photographer's name and the caption out of it. The title, dates and tags come straight from the JSON.
3. **Download.** Each image is downloaded with `requests.get()`, checked with `raise_for_status()` and saved in 100,000-byte chunks with `iter_content()`. The program waits one second between downloads so it doesn't overload Flickr's servers.
4. **Save a CSV.** One row per photo: the extracted text, the image's URL, the saved file's name and whether the download worked.

To search for a different category, change `CATEGORY` in the next cell and run all the cells again. The program needs `requests`, `beautifulsoup4` and `pandas`, which come with Anaconda and Google Colab (otherwise install them with `pip install requests beautifulsoup4 pandas`).

In [ ]:
import html
import json
import os
import re
import time

import bs4
import pandas as pd
import requests
from IPython.display import Image, display

CATEGORY = 'sunset'   # the category to search for; separate several tags with commas

FEED_URL = 'https://www.flickr.com/services/feeds/photos_public.gne'
HEADERS = {'User-Agent': 'Mozilla/5.0 (compatible; Week9ImageDownloader/1.0; Python requests)'}

# Flickr tags are lowercase with no spaces ("Golden Retriever" is stored as "goldenretriever").
tags = ','.join(tag.strip().lower().replace(' ', '') for tag in CATEGORY.split(',') if tag.strip())
if not tags:
    raise ValueError('Set CATEGORY to something to search for, such as "sunset".')

folder = 'flickr_' + re.sub(r'[^\w-]+', '_', tags)   # where the images go, e.g. flickr_sunset
csv_path = folder + '_images.csv'                     # e.g. flickr_sunset_images.csv
os.makedirs(folder, exist_ok=True)

print('Searching Flickr for photos tagged:', tags)
print('Images will be saved in:', os.path.abspath(folder))

## Step 1: Search Flickr for the category

The tag goes in the feed's query string. `format=json` asks for JSON instead of the default Atom XML, and `nojsoncallback=1` returns plain JSON rather than JSON wrapped in a JavaScript function call.

In [ ]:
params = {'tags': tags, 'tagmode': 'all', 'format': 'json', 'nojsoncallback': 1}
res = requests.get(FEED_URL, params=params, headers=HEADERS, timeout=30)
res.raise_for_status()
res.encoding = 'utf-8'   # the feed is UTF-8, so titles in any language decode correctly

try:
    feed = json.loads(res.text)
except json.JSONDecodeError:
    # Flickr writes apostrophes as \' which isn't valid JSON, so fix those and parse again.
    feed = json.loads(res.text.replace("\\'", "'"), strict=False)

items = feed.get('items', [])
print('Requested:', res.url)
print(f'Found {len(items)} photos.')
if not items:
    print('No photos found. Try a different CATEGORY.')

## Step 2: Extract the text for each photo

Each item's `description` is HTML that looks like this:

```html
<p><a href="https://www.flickr.com/people/...">Photographer</a> posted a photo:</p>
<p><a href="..." title="..."><img src="..._m.jpg" ... /></a></p>
<p>The photographer's caption, if there is one</p>
```

Beautiful Soup finds the photographer's link in the first paragraph, and any paragraphs after the thumbnail hold the caption.

In [ ]:
photos = []
for number, item in enumerate(items, start=1):
    soup = bs4.BeautifulSoup(item.get('description') or '', 'html.parser')
    paragraphs = soup.select('p')
    name_link = paragraphs[0].select_one('a') if paragraphs else None

    title = html.unescape(item.get('title') or '').strip()   # html.unescape turns &amp; back into &
    photographer = name_link.get_text(strip=True) if name_link else ''
    caption = ' '.join(p.get_text(' ', strip=True) for p in paragraphs[2:])

    photos.append({
        'number': number,
        'category': tags,
        'title': title,
        'photographer': photographer,
        'date_taken': item.get('date_taken', ''),
        'uploaded': item.get('published', ''),
        'tags': item.get('tags', ''),
        'caption': caption,
        'photo_page': item.get('link', ''),
        'small_image_url': (item.get('media') or {}).get('m', ''),
    })
    print(f'{number:2}. {title or "(untitled)"} by {photographer or "unknown"}')

## Step 3: Download every image

The feed links to a small, 240-pixel version of each photo whose URL ends in `_m.jpg`. Changing that ending to `_b.jpg` gives the same photo at 1,024 pixels on its long side, so the program downloads that version first and falls back to the small one if the large one isn't available.

In [ ]:
def download_image(url, file_path):
    """Download one image and save it to file_path in 100,000-byte chunks (Chapter 13)."""
    res = requests.get(url, headers=HEADERS, timeout=30)
    res.raise_for_status()
    # If a photo or size isn't available, Flickr can redirect to a "photo unavailable" placeholder.
    if 'photo_unavailable' in res.url or not res.headers.get('Content-Type', '').startswith('image/'):
        raise ValueError('Flickr has no image at this address')
    with open(file_path, 'wb') as image_file:
        for chunk in res.iter_content(100000):
            image_file.write(chunk)


for photo in photos:
    photo.update({'image_url': '', 'file_name': '', 'file_size_kb': None, 'status': 'failed'})
    small_url = photo['small_image_url']
    large_url = small_url.replace('_m.', '_b.')
    urls_to_try = [large_url, small_url] if large_url != small_url else [small_url]

    for url in urls_to_try:
        file_path = os.path.join(folder, f"{photo['number']:02}_{os.path.basename(url)}")
        print(f"Downloading image {photo['number']} of {len(photos)}: {url}")
        try:
            download_image(url, file_path)
        except Exception as exc:
            print('    There was a problem:', exc)
            continue
        photo['image_url'] = url
        photo['file_name'] = file_path
        photo['file_size_kb'] = round(os.path.getsize(file_path) / 1024, 1)
        photo['status'] = 'downloaded' if url == large_url else 'downloaded (small version)'
        break

    time.sleep(1)   # pause so we don't hammer Flickr's servers

downloaded = [photo for photo in photos if photo['status'].startswith('downloaded')]
print(f'\nDownloaded {len(downloaded)} of {len(photos)} images into {os.path.abspath(folder)}')

## Step 4: Save the results to a CSV file

The CSV is saved in the same folder as this notebook. It's written as `utf-8-sig` so Excel shows titles in other languages, and emoji, correctly.

In [ ]:
columns = ['number', 'category', 'title', 'photographer', 'date_taken', 'uploaded', 'tags',
           'caption', 'photo_page', 'image_url', 'file_name', 'file_size_kb', 'status']
results = pd.DataFrame(photos, columns=columns)
results.to_csv(csv_path, index=False, encoding='utf-8-sig')

print(f'Saved {len(results)} rows to {os.path.abspath(csv_path)}')
results[['number', 'title', 'photographer', 'date_taken', 'file_name', 'status']]

## Step 5: Preview some of the downloaded images

In [ ]:
if not downloaded:
    print('No images were downloaded.')
for photo in downloaded[:3]:
    print(f'{photo["title"] or "(untitled)"} by {photo["photographer"] or "unknown"}')
    display(Image(filename=photo['file_name'], width=320))

## References

Flickr. "Public Photos & Video." The App Garden. Accessed November 2, 2026. https://www.flickr.com/services/feeds/docs/photos_public/.

Sweigart, Al. "Web Scraping." Chap. 13 in *Automate the Boring Stuff with Python: Practical Programming for Total Beginners*. 3rd ed. San Francisco: No Starch Press, 2025. https://automatetheboringstuff.com/3e/chapter13.html.